### Importar Pacotes

In [1]:
from  utils import logging, date, json, re, BeautifulSoup, requests,math,pd
from pathlib import Path
import sqlite3
import os
# Configuração do logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Ignorar avisos de certificado SSL (não recomendado para produção)
import urllib3
from urllib.parse import unquote, urlparse, parse_qs
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

### Idealista

https://www.idealista.pt/comprar-casas/lisboa/mapa \
Zona de Lisboa > Benfica
url: https://www.idealista.pt/comprar-casas/lisboa/benfica/mapa \
Zonas em Benfica
- Portas de Benfica - Mercado de Benfica 124
- Bairro de Santa Cruz 119
- Avenida do Uruguai 100
- Fonte Nova - Calhariz 99
- Calçada do Tojal 93
- Arneiros 40
- Igreja de Benfica 37
- Pedralvas 13
- Charquinho 9

Distrito > Concelho > Zona > pagina
https://www.idealista.pt/comprar-casas/benfica/portas-de-benfica-mercado-de-benfica/pagina-1

com filtros
https://www.idealista.pt/comprar-casas/lisboa/alvalade/com-preco-max_400000,preco-min_100000,tamanho-min_40,tamanho-max_200,ultimo-andar,andares-intermedios,res-do-chao/
 

In [ ]:
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def download_imovirtual(url, filename="imovirtual.html"):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/117.0",
        "Accept-Language": "pt-PT,pt;q=0.9,en-US;q=0.8,en;q=0.7",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }

    r = requests.get(url, headers=headers, timeout=30, verify=False)
    r.raise_for_status()

    with open(filename, "w", encoding="utf-8") as f:
        f.write(r.text)

    print(f"✅ Guardado em {filename}")


url = "https://www.imovirtual.com/pt/resultados/comprar/apartamento/lisboa/amadora/?page=1"
download_imovirtual(url,filename='imovirtual_amadora.html')

✅ Guardado em imovirtual_amadora.html


In [3]:
import requests, json, re
from bs4 import BeautifulSoup
import certifi

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "pt-PT,pt;q=0.9,en;q=0.8"
}

def fetch_json_ld(url, timeout=30):
    """Faz GET ao URL, não salva HTML, extrai todos os blocos application/ld+json e devolve lista de dicts."""
    r = requests.get(url, headers=HEADERS, timeout=timeout, verify=False)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    jsonld = []
    for s in soup.find_all("script", {"type":"application/ld+json"}):
        if not s.string:
            continue
        text = s.string.strip()
        try:
            data = json.loads(text)
            # normalizar listas/dicts
            if isinstance(data, list):
                jsonld.extend(data)
            else:
                jsonld.append(data)
        except Exception as e:
            # tentativa de limpeza leve (algumas páginas têm JS-like objects)
            try:
                cleaned = re.sub(r'(\w+):', r'"\1":', text)  # tentativa simples (cuidado)
                data = json.loads(cleaned)
                if isinstance(data, list):
                    jsonld.extend(data)
                else:
                    jsonld.append(data)
            except Exception:
                # se falhar, ignorar esse bloco
                continue
    return jsonld

# Exemplo de uso:
blocks = fetch_json_ld("https://www.imovirtual.com/pt/resultados/comprar/apartamento/lisboa/amadora/?page=1")
print(len(blocks), blocks[0])

c:\Users\XB926SS\OneDrive - EY\Documents\GitHub\house-price-tracker\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.imovirtual.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\XB926SS\OneDrive - EY\Documents\GitHub\house-price-tracker\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.imovirtual.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


1 {'@context': 'https://schema.org', '@graph': [{'@type': 'WebPage', 'url': 'https://www.imovirtual.com/pt/resultados/comprar/apartamento/lisboa/amadora', 'isPartOf': {'@type': 'WebSite', 'name': 'Imovirtual', 'url': 'https://www.imovirtual.com', 'description': 'Encontre a sua casa de sonho com o Imovirtual. Disponibilizamos uma seleção personalizada de apartamentos, moradias, terrenos, imóveis comerciais para comprar e para arrendar. Ofertas de promotores, agências imobiliárias e diretamente com os proprietários.', 'inLanguage': 'pt'}, 'about': {'@type': 'Organization', 'foundingDate': '2011', 'logo': 'https://statics.imovirtual.com/fp_statics/images/logo/imovirtual2.svg', 'brand': 'OLX Group', 'alternateName': ['imovirtual', 'imovirtual.com', 'imovirtual olx', 'imovirtual lisboa', 'imovirtual porto', 'imovirtual pt', 'imovirtual aveiro', 'www.imovirtual.com', 'imovirtual com', 'imovirtual casa'], 'sameAs': ['https://www.facebook.com/Imovirtual', 'https://www.instagram.com/imovirtual/